In [1]:
import numpy as np

zt = np.array([5.0, 3.0, 2.0, 0.0])
zs = np.array([3.0, 2.5, 1.0, 0.5])
y = np.array([1.0, 0.0, 0.0, 0.0])
words = ["hospital", "clinic", "ward", "school"]
T = 2.0
alpha = 0.5
lr = 0.5

def softmax(z, temp=1.0):
    z = np.array(z, dtype=float) / temp
    z = z - z.max()
    e = np.exp(z)
    return e / e.sum()

def kl(p, q):
    return float(np.sum(p * np.log(p / q)))

def ce(label, q):
    return float(-np.sum(label * np.log(q)))

In [2]:
p1 = softmax(zt, 1.0)
q1 = softmax(zs, 1.0)
print("part (a)")
print("word      teacher   student")
for w, a, b in zip(words, p1, q1):
    print(f"{w:9s} {a:.4f}    {b:.4f}")
print("sum", p1.sum(), q1.sum())

part (a)
word      teacher   student
hospital  0.8390    0.5483
clinic    0.1135    0.3325
ward      0.0418    0.0742
school    0.0057    0.0450
sum 1.0000000000000002 1.0000000000000002


In [3]:
print("part (b)")
print("teacher logits / T:", zt / T)
print("student logits / T:", zs / T)
pT = softmax(zt, T)
qT = softmax(zs, T)
print("word      teacher   student")
for w, a, b in zip(words, pT, qT):
    print(f"{w:9s} {a:.4f}    {b:.4f}")
print("sum", pT.sum(), qT.sum())

part (b)
teacher logits / T: [2.5 1.5 1.  0. ]
student logits / T: [1.5  1.25 0.5  0.25]
word      teacher   student
hospital  0.5977    0.4110
clinic    0.2199    0.3201
ward      0.1334    0.1512
school    0.0491    0.1177
sum 1.0 0.9999999999999999


In [4]:
print("part (c)")
terms = pT * np.log(pT / qT)
for w, t in zip(words, terms):
    print(f"{w:9s} {t:.5f}")
klv = kl(pT, qT)
print("KL(pT || qT) =", round(klv, 5))

part (c)
hospital  0.22385
clinic    -0.08256
ward      -0.01673
school    -0.04295
KL(pT || qT) = 0.08161


In [5]:
print("part (d)")
cev = ce(y, q1)
print("CE = -ln(q1[hospital]) = -ln(%.5f) = %.5f" % (q1[0], cev))

part (d)
CE = -ln(q1[hospital]) = -ln(0.54826) = 0.60100


In [6]:
print("part (e)")
soft_term = alpha * T**2 * klv
hard_term = (1 - alpha) * cev
total = soft_term + hard_term
print("alpha * T^2 * KL =", round(soft_term, 5))
print("(1 - alpha) * CE =", round(hard_term, 5))
print("L =", round(total, 5))

part (e)
alpha * T^2 * KL = 0.16321
(1 - alpha) * CE = 0.3005
L = 0.46372


In [7]:
print("part (f)")
soft_grad = alpha * T * (qT - pT)
hard_grad = (1 - alpha) * (q1 - y)
grad = soft_grad + hard_grad
print("soft part:", np.round(soft_grad, 5))
print("hard part:", np.round(hard_grad, 5))
print("gradient :", np.round(grad, 5))

def total_loss(z):
    return alpha * T**2 * kl(softmax(zt, T), softmax(z, T)) + (1 - alpha) * ce(y, softmax(z, 1.0))

eps = 1e-6
num = np.zeros(4)
for i in range(4):
    a = zs.copy()
    b = zs.copy()
    a[i] += eps
    b[i] -= eps
    num[i] = (total_loss(a) - total_loss(b)) / (2 * eps)
print("finite difference check:", np.round(num, 5))

part (f)
soft part: [-0.18671  0.10019  0.01783  0.06869]
hard part: [-0.22587  0.16627  0.0371   0.0225 ]
gradient : [-0.41258  0.26646  0.05493  0.09119]
finite difference check: [-0.41258  0.26646  0.05493  0.09119]


In [8]:
print("part (g)")
zs_new = zs - lr * grad
print("new logits:", np.round(zs_new, 5))
print("new student T=1:", np.round(softmax(zs_new, 1.0), 5))
print("new student T=2:", np.round(softmax(zs_new, T), 5))
new_loss = total_loss(zs_new)
print("loss before:", round(total, 5))
print("loss after :", round(new_loss, 5))
print("drop       :", round(total - new_loss, 5))

part (g)
new logits: [3.20629 2.36677 0.97254 0.45441]
new student T=1: [0.62389 0.26947 0.06683 0.03981]
new student T=2: [0.44701 0.29377 0.14631 0.11291]
loss before: 0.46372
loss after : 0.34926
drop       : 0.11446
